# Monitoring | Helper compartilhado

Funções usadas por Silver, Gold e DQ:

- `iniciar_etapa()` / `registrar_execucao(...)`: grava duração, linhas e status em `workspace.monitoring.pipeline_runs` (append).
- `RegistroDQ`: grava cada regra em `workspace.monitoring.qualidade_resultados` (append) e só então decide o gate.
- `RegistroDQ.gate()`: levanta `RuntimeError` se houver falha BLOQUEANTE. A exceção **não** é capturada aqui.

Uso: `%run ../../Monitoring/_common_monitoring` (ajustar o número de `../` conforme a pasta do notebook).

In [0]:
import time
import uuid
from datetime import datetime, timezone

LIMIAR_CORRESPONDENCIA_PADRAO = 95.0  # % mínimo de linhas com par em um join (ver README, seção DQ)

_SCHEMA_RUNS = """
    run_id string, layer string, entity string, started_at timestamp,
    finished_at timestamp, duration_seconds double, status string,
    row_count long, error_message string
"""

_SCHEMA_DQ = """
    execution_id string, layer string, table_name string, rule_name string,
    severity string, checked_at timestamp, failed_rows long, status string, details string
"""


def _agora():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def garantir_tabelas_monitoramento():
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.monitoring")
    spark.sql("""
        CREATE TABLE IF NOT EXISTS workspace.monitoring.pipeline_runs (
            run_id STRING, layer STRING, entity STRING, started_at TIMESTAMP,
            finished_at TIMESTAMP, duration_seconds DOUBLE, status STRING,
            row_count BIGINT, error_message STRING
        ) USING DELTA
    """)
    spark.sql("""
        CREATE TABLE IF NOT EXISTS workspace.monitoring.qualidade_resultados (
            execution_id STRING, layer STRING, table_name STRING, rule_name STRING,
            severity STRING, checked_at TIMESTAMP, failed_rows BIGINT, status STRING, details STRING
        ) USING DELTA
    """)


def iniciar_etapa():
    """Retorna o marcador de início a ser passado para registrar_execucao()."""
    garantir_tabelas_monitoramento()
    return {"started": _agora(), "tick": time.perf_counter(), "run_id": str(uuid.uuid4())}


def registrar_execucao(layer, entity, marcador, status="SUCESSO", row_count=None, erro=None):
    """Acrescenta uma linha em pipeline_runs (nunca sobrescreve)."""
    finished = _agora()
    duration = time.perf_counter() - marcador["tick"]
    linha = [(marcador["run_id"], layer, entity, marcador["started"], finished,
              float(duration), status, None if row_count is None else int(row_count),
              None if erro is None else str(erro)[:2000])]
    (spark.createDataFrame(linha, _SCHEMA_RUNS)
        .write.format("delta").mode("append")
        .saveAsTable("workspace.monitoring.pipeline_runs"))
    print(f"[monitoring] {layer}/{entity}: {status} em {duration:.1f}s, linhas={row_count}")


def contar_tabela(nome):
    return spark.table(nome).count()


class RegistroDQ:
    """Acumula resultados de regras, persiste cada um e aplica o gate no final."""

    def __init__(self, layer, limiar_correspondencia=LIMIAR_CORRESPONDENCIA_PADRAO):
        garantir_tabelas_monitoramento()
        self.layer = layer
        self.limiar = float(limiar_correspondencia)
        self.execution_id = str(uuid.uuid4())
        self.checked_at = _agora()
        self.bloqueios = []

    def registrar(self, table_name, rule_name, severity, failed_rows, details=""):
        failed_rows = int(failed_rows)
        status = "FALHOU" if failed_rows > 0 else "PASSOU"
        (spark.createDataFrame(
            [(self.execution_id, self.layer, table_name, rule_name, severity,
              self.checked_at, failed_rows, status, str(details)[:4000])], _SCHEMA_DQ)
            .write.format("delta").mode("append")
            .saveAsTable("workspace.monitoring.qualidade_resultados"))
        if status == "FALHOU" and severity == "BLOQUEANTE":
            self.bloqueios.append(f"{table_name}:{rule_name}={failed_rows}")
        print(f"{table_name} | {rule_name} | {severity} | {status} | falhas={failed_rows}")

    def chave_nula_e_duplicada(self, full_table, chaves, severity="BLOQUEANTE"):
        cond_nula = " OR ".join(f"`{k}` IS NULL" for k in chaves)
        nulas = spark.sql(f"SELECT COUNT(*) n FROM {full_table} WHERE {cond_nula}").first()["n"]
        lista = ", ".join(f"`{k}`" for k in chaves)
        dups = spark.sql(
            f"SELECT COUNT(*) n FROM (SELECT {lista} FROM {full_table} "
            f"GROUP BY {lista} HAVING COUNT(*) > 1)").first()["n"]
        det = f"Chave de negócio: {', '.join(chaves)}"
        self.registrar(full_table, "CHAVE_NULA", severity, nulas, det)
        self.registrar(full_table, "CHAVE_DUPLICADA", severity, dups, det)

    def taxa_correspondencia(self, nome_join, tabela_esq, tabela_dir, condicao, severity="BLOQUEANTE"):
        """Mede quantas linhas da esquerda encontram par. Reprova abaixo do limiar.

        tabela_esq/tabela_dir podem ser nomes de tabela ou subqueries entre parênteses.
        `condicao` usa os aliases l (esquerda) e r (direita).
        Retorna a taxa em %.
        """
        total = spark.sql(f"SELECT COUNT(*) t FROM {tabela_esq}").first()["t"]
        sem_par = spark.sql(
            f"SELECT COUNT(*) t FROM {tabela_esq} l LEFT ANTI JOIN {tabela_dir} r ON {condicao}"
        ).first()["t"]
        com_par = total - sem_par
        taxa = (com_par / total * 100.0) if total > 0 else 0.0  # tabela vazia NÃO aprova
        reprovou = taxa < self.limiar
        self.registrar(
            nome_join, "TAXA_CORRESPONDENCIA_JOIN", severity,
            sem_par if reprovou else 0,
            f"Taxa: {taxa:.2f}% ({com_par}/{total}). Limiar: {self.limiar}%. Sem par: {sem_par}.")
        return taxa

    def gate(self):
        """Interrompe a execução se houver falha bloqueante (chamar DEPOIS de registrar tudo)."""
        print(f"Execução DQ {self.layer}: {self.execution_id} | bloqueios: {len(self.bloqueios)}")
        if self.bloqueios:
            raise RuntimeError(f"DQ {self.layer} bloqueada: " + "; ".join(self.bloqueios))
        print(f"DQ {self.layer} concluída. Gate liberado.")